# Self‑Driving Car 3D Simulator with CNN
Lane detection + CNN + Q‑Learning control.

**Controls**
- P → Pause / Resume
- Q → Stop simulation


## Imports and Environment Setup

In [1]:
import cv2
import mss
import numpy as np
import os
import pyautogui as p
import random
import keyboard
import time

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Input
from tensorflow.keras.optimizers import SGD

GPU = 0
if GPU == 0:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
else:
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print('Dependencies loaded')

Dependencies loaded


## Fast Screen Capture Setup

In [2]:
sct = mss.mss()

game_area = {
 'top':230,
 'left':330,
 'width':1180,
 'height':660
}

print('Screen capture ready')

Screen capture ready


## Lane Detection Function

In [3]:
def CalculateLanes(OrgImage):
    try:
        HSLImg = cv2.cvtColor(OrgImage, cv2.COLOR_BGR2HLS)
        lower = np.uint8([10,0,100])
        upper = np.uint8([40,255,255])
        yellow_mask = cv2.inRange(HSLImg, lower, upper)
        YellowImg = cv2.bitwise_and(OrgImage, OrgImage, mask=yellow_mask)

        GrayImg = cv2.cvtColor(YellowImg, cv2.COLOR_BGR2GRAY)
        blurredImg = cv2.GaussianBlur(GrayImg,(5,5),0)
        imageWithEdges = cv2.Canny(blurredImg,200,700)

        points = np.array([[0,310],[0,300],[220,210],[380,210],[600,300],[600,310]])
        mask = np.zeros_like(imageWithEdges)
        cv2.fillPoly(mask,[points],255)
        croppedImg = cv2.bitwise_and(blurredImg,mask)

        lines = cv2.HoughLinesP(croppedImg,1,np.pi/180,180,np.array([]),180,5)

        left_lines=[]
        right_lines=[]
        length_left=[]
        length_right=[]

        if lines is not None:
            for line in lines:
                for x1,y1,x2,y2 in line:
                    if x1==x2 or y1==y2:
                        continue
                    slope=(y2-y1)/(x2-x1)
                    intercept=y1-slope*x1
                    length=np.sqrt((y2-y1)**2+(x2-x1)**2)

                    if slope<0:
                        left_lines.append((slope,intercept))
                        length_left.append(length)
                    else:
                        right_lines.append((slope,intercept))
                        length_right.append(length)

        left_lane = np.dot(length_left,left_lines)/np.sum(length_left) if len(length_left)>0 else None
        right_lane = np.dot(length_right,right_lines)/np.sum(length_right) if len(length_right)>0 else None

        if left_lane is not None and right_lane is not None:
            y1_coord=croppedImg.shape[0]
            y2_coord=int(y1_coord*0.6)

            lx1=int((y1_coord-left_lane[1])/left_lane[0])
            lx2=int((y2_coord-left_lane[1])/left_lane[0])
            rx1=int((y1_coord-right_lane[1])/right_lane[0])
            rx2=int((y2_coord-right_lane[1])/right_lane[0])

            emptImg=np.zeros_like(OrgImage)
            cv2.line(emptImg,(lx1,y1_coord),(lx2,y2_coord),(255,0,0),20)
            cv2.line(emptImg,(rx1,y1_coord),(rx2,y2_coord),(255,0,0),20)

            finalImg=cv2.addWeighted(OrgImage,1.0,emptImg,0.95,0.0)
            return finalImg,False
        else:
            return OrgImage,True
    except:
        return OrgImage,True


## Frame Capture + CNN Preprocessing

In [4]:
def getFrames():
    gameImg = np.array(sct.grab(game_area))
    gameImg = cv2.resize(gameImg, (600,400), interpolation=cv2.INTER_AREA)

    img, errors = CalculateLanes(gameImg)

    cv2.imshow('CNN View', img)
    cv2.waitKey(1)

    img_cnn = cv2.cvtColor(cv2.resize(img,(84,84)), cv2.COLOR_BGR2GRAY)
    img_cnn = img_cnn.astype(np.float32) / 255.0
    img_cnn = img_cnn.reshape(1,84,84)

    input_img = np.stack((img_cnn,img_cnn,img_cnn,img_cnn), axis=3)

    return input_img, errors

## Car Control Functions

In [5]:
# Control functions using PyAutoGUI
def straight():
    p.keyDown("up")
    p.keyUp("right")
    p.keyUp("left")
    p.keyUp("up")

def right():
    p.keyDown("right")
    p.keyUp("right")

def left():
    p.keyDown("left")
    p.keyUp("left")

print("Control functions (straight, right, left) defined.")

Control functions (straight, right, left) defined.


## CNN Model

In [6]:
moves=3
learningRate=0.00025

model = Sequential([
Input(shape=(84,84,4)),
Conv2D(32,(3,3),padding='same',activation='relu'),
Conv2D(64,(3,3),activation='relu'),
Conv2D(64,(3,3),activation='relu'),
Flatten(),
Dense(512,activation='relu'),
Dense(3,activation='linear')
])

model.compile(loss='mse',optimizer=SGD(learning_rate=learningRate))

print('CNN Ready')

CNN Ready


## Main Simulation Loop (Pause / Stop Enabled)

In [7]:
paused = False
reward = 0
last_reward = -1

print("Press P to pause/resume | Q to stop")

input_img, errors = getFrames()

while True:

    if keyboard.is_pressed('p'):
        paused = not paused
        print("Paused" if paused else "Resumed")
        time.sleep(0.5)

    if keyboard.is_pressed('q'):
        print("Stopping simulation...")
        break

    if paused:
        continue

    output = model.predict(input_img, verbose=0)
    action = np.argmax(output[0])

    if action == 0:
        straight()
    elif action == 1:
        right()
    else:
        left()

    input_img, errors = getFrames()

    if not errors:
        reward += 1

    if reward != last_reward:
        print("Score:", reward)
        last_reward = reward

    time.sleep(0.03)    

cv2.destroyAllWindows()
print("Final Score:", reward)

Press P to pause/resume | Q to stop
Score: 0
Score: 1
Score: 2
Score: 3
Score: 4
Score: 5
Score: 6
Score: 7
Score: 8
Score: 9
Score: 10
Score: 11
Score: 12
Score: 13
Score: 14
Score: 15
Score: 16
Score: 17
Score: 18
Score: 19
Score: 20
Score: 21
Score: 22
Score: 23
Score: 24
Score: 25
Score: 26
Score: 27
Score: 28
Paused
Resumed
Stopping simulation...
Final Score: 28
